# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata
md = dataset.metadata

print(f"{md.name}: {md.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

The Croissant dataset organizes tabular data into record sets. Each record set is uniquely identified by its `@id`. Fields and columns within a record set are also uniquely referenced by their `@id`s—these will be used for subsequent data extraction.

Let's enumerate all record sets, then for each record set, enumerate its fields and columns (by their `@id` and name if available).

In [ ]:
# List all record sets in the dataset
print('--- Record Sets in Dataset ---')
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"@id: {rs['@id']}, name: {rs.get('name', '(no name)')}")

# For each record set, list fields and columns by @id
for rs in record_sets:
    print(f"\nRecordSet: {rs['@id']}")
    fields = rs.get('field', [])
    if not isinstance(fields, list):
        fields = [fields]
    print("  Fields:")
    for field in fields:
        # In Croissant, field entries may be strings or dicts
        f_id = field if isinstance(field, str) else field.get('@id', None)
        print(f"    - {f_id}")
    columns = rs.get('column', [])
    if columns:
        if not isinstance(columns, list):
            columns = [columns]
        print("  Columns:")
        for col in columns:
            c_id = col if isinstance(col, str) else col.get('@id', None)
            print(f"    - {c_id}")

## 3. Data Extraction
Load data from specific record sets using their `@id`s.

1. Select record set `@id`s from the overview above.
2. Extract data from each record set into a `pandas.DataFrame`.
3. Display available columns for one key record set.

In [ ]:
# Prepare to extract all available record sets
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:  # Only add if not empty
        dataframes[record_set_id] = pd.DataFrame(records)

if dataframes:
    # Pick the first record_set for display
    main_rs_id = list(dataframes.keys())[0]
    print(f"Columns in record set {main_rs_id}: {dataframes[main_rs_id].columns.tolist()}")
    display(dataframes[main_rs_id].head())
else:
    print("No tabular record sets found in this dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply data preparation and analysis steps:
- Filter records by a numeric field.
- Normalize this numeric field.
- Group and aggregate based on another field.

**Note:** Replace `<numeric_field_id>` and `<group_field_id>` below with actual field IDs available in your dataset (see outputs above).

In [ ]:
import numpy as np

# Select the main record set
if dataframes:
    rs_id = main_rs_id
    df = dataframes[rs_id]

    # Choose a numeric field by @id or column name (update this as appropriate)
    # Example fallback: use the first numeric column found
    numeric_field_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_field_candidates:
        numeric_field_id = numeric_field_candidates[0]
        print(f"Using numeric field: {numeric_field_id}")

        threshold = df[numeric_field_id].mean()  # Example threshold: mean value
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records in {rs_id} where {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by another field (categorical), fallback to second column
        non_numeric_fields = [col for col in df.columns if df[col].dtype == object and col != numeric_field_id]
        if non_numeric_fields:
            group_field = non_numeric_fields[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].agg(['mean', 'count'])
            print(f"Grouped data by {group_field}:")
            display(grouped_df.head())
        else:
            print("No suitable non-numeric field found for grouping.")
    else:
        print("No numeric fields available for analysis.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize the distribution of the selected numeric field or show relationships grouped by another attribute, if data is available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and 'numeric_field_id' in locals():
    # Histogram of the numeric field
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If grouping field exists, plot boxplot
    if 'group_field' in locals() and group_field in df.columns:
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to:
- Load FAIR² dataset metadata and records via Croissant schema and `mlcroissant`.
- List available record sets, fields, and columns by their `@id`.
- Extract tabular data into pandas DataFrames.
- Perform simple EDA with filtering, normalization, and grouping, referencing all entities by their `@id`.
- Visualize distributions of key numeric variables, supporting further analysis for policy or research use.